In [1]:
import pandas as pd
import numpy as np
import json
import gensim.corpora as corpora
import pyLDAvis.gensim 

In [2]:
#data = pd.read_csv('G:\Github\Recipe_reco\data\RAW_recipes.csv')
data=pd.read_json(r'C:\Users\bmhardyman\OneDrive - UNIMAS\Kemas Wiharja\Data\aziekitchen-udang.jl', lines = True)

In [ ]:
data.head()

In [ ]:
#yang simpan resipi.
#TODO cleaning data untuk buat stopwords dalam bahasa indo/malay. boleh guna senarai dari bahasa indo
data['page_description']

In [ ]:
#boleh run sekiranya ingin masukkan column lain dari dataset, kalo ga ya jangan run donk
convert_to_list = ['page_description']

for c in convert_to_list:
    data[c] = data[c].apply(lambda x: eval(x))

In [ ]:
#just check aja
data.info()

In [3]:
#save dalam acar biar ngebut loading ntar.
data.to_pickle(r'C:\Users\bmhardyman\OneDrive - UNIMAS\Kemas Wiharja\Data\raw_df.pkl')

In [4]:
def treat_ingredients(ing_list):
    output = []
    for ingredient in ing_list:
        ingredient_list = ingredient.split(' ')
        output.append("_".join(ingredient_list))
    return output
ingredients_all = data.page_description.apply(lambda x: treat_ingredients(x))

In [5]:
import gensim
from sklearn.feature_extraction.text import CountVectorizer

# Load the list of documents
ingredients_all = data.page_description.apply(lambda x: ", ".join(x))

# Use CountVectorizor to find three letter tokens, remove stop_words, 
# remove tokens that don't appear in at least 20 documents,
# remove tokens that appear in more than 20% of the documents
vect = CountVectorizer(token_pattern='(?u)\\b\\w\\w\\w+\\b')

# Fit and transform
X = vect.fit_transform(ingredients_all)

# Convert sparse matrix to gensim corpus.
corpus = gensim.matutils.Sparse2Corpus(X, documents_columns=False)

# Mapping from word IDs to words (To be used in LdaModel's id2word parameter)
id_map = dict((v, k) for k, v in vect.vocabulary_.items())


In [28]:
#fit LDA model
#ldamodel = gensim.models.ldamodel.LdaModel(corpus,num_topics = 5,passes = 20, random_state = 0, id2word = id_map)
ldamodel = gensim.models.ldamodel.LdaModel(corpus,num_topics = 5,passes = 20, random_state = 0, id2word = id_map,chunksize=100,alpha='auto',per_word_topics=True)

In [29]:
ldamodel.print_topics(num_topics = 5,num_words = 10)
#print(ldamodel.print_topics())
#doc_lda = ldamodel[corpus]

[(0,
  '0.013*"aishah" + 0.013*"aduhaai" + 0.013*"300" + 0.013*"1kg" + 0.005*"agaklah" + 0.005*"aisyah" + 0.003*"bagaimana" + 0.002*"alahaaai" + 0.002*"1000" + 0.002*"berbatu"'),
 (1,
  '0.017*"aduhaai" + 0.015*"1kg" + 0.013*"300" + 0.009*"aishah" + 0.006*"adil" + 0.006*"200" + 0.005*"agaklah" + 0.004*"abadi" + 0.004*"beli" + 0.004*"ada"'),
 (2,
  '0.010*"1kg" + 0.010*"aduhaai" + 0.009*"300" + 0.007*"aishah" + 0.004*"ada" + 0.004*"adil" + 0.004*"200" + 0.004*"agaklah" + 0.003*"adohaiii" + 0.003*"10x"'),
 (3,
  '0.011*"1kg" + 0.011*"300" + 0.011*"aduhaai" + 0.009*"ahmad" + 0.005*"200" + 0.004*"agaklah" + 0.004*"adil" + 0.004*"ibrahim" + 0.003*"ada" + 0.003*"keperangan"'),
 (4,
  '0.126*"akhirnya" + 0.009*"1kg" + 0.009*"aduhaai" + 0.008*"ada" + 0.007*"300" + 0.005*"aishah" + 0.003*"ahmad" + 0.003*"adil" + 0.003*"beraya" + 0.003*"berfikiran"')]

In [8]:
for i in ldamodel.get_document_topics(corpus):
    print(i)

[(0, 0.4393415), (2, 0.27433598), (4, 0.28617597)]
[(0, 0.9995263)]
[(0, 0.99943006)]
[(0, 0.86301476), (4, 0.13666062)]
[(0, 0.037598904), (1, 0.8370618), (2, 0.12505075)]
[(0, 0.7328162), (2, 0.2250382), (4, 0.041921046)]
[(0, 0.4267413), (4, 0.5728322)]
[(0, 0.9994475)]
[(0, 0.43160018), (1, 0.083472475), (2, 0.48470506)]
[(0, 0.735214), (2, 0.11165249), (4, 0.15297298)]
[(0, 0.33455768), (2, 0.53825665), (4, 0.12701592)]
[(0, 0.8924723), (2, 0.035364807), (4, 0.071955025)]
[(0, 0.30604658), (2, 0.4139469), (4, 0.27983162)]
[(0, 0.83811563), (4, 0.1614784)]
[(0, 0.67893517), (1, 0.088929676), (2, 0.23185988)]
[(0, 0.5187973), (1, 0.2096946), (4, 0.27126852)]
[(0, 0.5413345), (2, 0.4054797), (4, 0.052935638)]
[(0, 0.08476092), (1, 0.9146181)]
[(0, 0.2393858), (4, 0.7602472)]
[(0, 0.99930817)]
[(0, 0.077895366), (4, 0.9219252)]
[(0, 0.4144393), (4, 0.58531827)]
[(0, 0.19366814), (2, 0.14322463), (4, 0.66294056)]
[(0, 0.77341735), (4, 0.22631395)]
[(0, 0.11597337), (4, 0.8837407)]
[(0,

In [22]:
word2id =dict((v, k) for k, v in vect.vocabulary_.items())
d = corpora.Dictionary()
d.id2token = id_map
d.token2id = word2id
#change id2word to d


In [26]:
pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim.prepare(ldamodel, corpus,id_map)
vis

AttributeError: 'dict_items' object has no attribute 'token2id'